In [ ]:
import os
#from azure.ai.ml import MLClient, Input, MpiDistribution, command
from azure.ai.ml import MLClient, Input, Output, PyTorchDistribution, command
from azure.ai.ml.entities import AmlCompute, Environment, BuildContext, Data
from azure.identity import DefaultAzureCredential
from azure.ai.ml.constants import AssetTypes
from dotenv import load_dotenv
load_dotenv(override=True)

# Azure ML workspace configuration
SUBSCRIPTION_ID = os.getenv("SUBSCRIPTION_ID")
RESOURCE_GROUP = os.getenv("RESOURCE_GROUP")
WORKSPACE_NAME = os.getenv("WORKSPACE_NAME")
COMPUTE_CLUSTER = "demo-gpucluster01"

# authentication via managed identity or service principal (no hard-coded creds)
ml_client = MLClient(DefaultAzureCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

# ensure compute cluster exists or create it
try:
    ml_client.compute.get(COMPUTE_CLUSTER)
except Exception:
    print("demo-gpucluster01 was not found")

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [5]:
from azure.ai.ml.entities import Data

# ローカルまたは Azure Storage 上のフォルダをデータセット登録
wiki_data = Data(
    name="wiki-indexed-dataset",
    version="1",
    type="uri_folder",
    path="https://demomlworkspac2559334577.blob.core.windows.net/azureml-blobstore-34281f7b-284f-4f46-ba74-89206cf6e29b/wiki-indexed-dataset/",
)
ml_client.data.create_or_update(wiki_data)

Data({'path': 'https://demomlworkspac2559334577.blob.core.windows.net/azureml-blobstore-34281f7b-284f-4f46-ba74-89206cf6e29b/wiki-indexed-dataset/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'wiki-indexed-dataset', 'description': None, 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/8f94592c-747a-4bb7-ab3e-0b569993c33c/resourceGroups/rg_amlworkspace/providers/Microsoft.MachineLearningServices/workspaces/demo-mlworkspace01/data/wiki-indexed-dataset/versions/1', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/demo-cpuinst01/code/Users/notanaha/kotoba-recipes-main', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x724a80dc5b10>, 'serialize': <msrest.serialization.Serializer object at 0x724a80dc59f0>, 'version': '1', 'latest_version': None, '

#### Docker environment

In [6]:
CLOUD_DIR = "./cloud"

In [ ]:
# job configuration
NUM_NODES = 1
NUM_GPU_PER_NODE = 1

In [ ]:
# define distributed training job
#dist = MpiDistribution()
#dist.process_count_per_instance = NUM_GPU_PER_NODE
dist = PyTorchDistribution(
    process_count_per_instance=NUM_GPU_PER_NODE,
    node_count=NUM_NODES
)

job = command(
    code="./cloud",
    command=(
        "python megatron_lm/tools/preprocess_data.py \
        --input ${{inputs.train_data}}/wikidump.jsonl \
        --output-prefix ${{outputs.indexed}}/wikidump \
        --tokenizer-type Llama2Tokenizer \
        --tokenizer-model ${{inputs.model_dir}} \
        --workers 1"
    ),
    inputs={
        "train_data": Input(
            type=AssetTypes.URI_FILE, 
            path="wiki_dump_02@latest"
        ),
        "model_dir": Input(
            type=AssetTypes.URI_FOLDER, 
            path="llama3-8b@latest"
        )
    },
    outputs={
        "indexed": Output(type=AssetTypes.URI_FOLDER,
                          path="azureml://datastores/workspaceblobstore/paths/wiki-indexed-dataset@latest",   # local → AzureML がアップロード
                          mode="rw_mount")                      # 次のジョブからマウント可能
    },
    environment="llama3-8b-wiki_env@latest",
    compute=COMPUTE_CLUSTER,
    instance_count=NUM_NODES,
    distribution=dist,
    environment_variables={
        "LOGLEVEL": "INFO",
        "NCCL_DEBUG": "WARN",
        "NCCL_DEBUG_SUBSYS": "WARN",
        "PYTHONFAULTHANDLER": "1",
        "CUDA_LAUNCH_BLOCKING": "0"
    },
    display_name="llama3-8b-wiki-index",
    experiment_name="llama3-8b-wiki-index-exp"
)

In [9]:
# submit the job
returned_job = ml_client.jobs.create_or_update(job)
print(f"Job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading cloud (0.96 MBs): 100%|█████

Job submitted: ivory_pot_4k44986ngv
Monitor at: https://ml.azure.com/runs/ivory_pot_4k44986ngv?wsid=/subscriptions/8f94592c-747a-4bb7-ab3e-0b569993c33c/resourcegroups/rg_amlworkspace/workspaces/demo-mlworkspace01&tid=16b3c013-d300-468d-ac64-7eda0820b6d3
